In [1]:
import pandas as pd
from pathlib import Path

INP = Path("../outputs/tables")

df = pd.read_csv(INP / "matches_raw.csv")
long_df = pd.read_csv(INP / "player_matches.csv")
features_agg = pd.read_csv(INP / "player_features.csv")

print(df.shape, long_df.shape, features_agg.shape)
df.head(2)


(66681, 17) (133362, 18) (622, 15)


,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Dosedel S.,Ljubicic I.,Dosedel S.,63,77,-1,-1,-1.0,-1.0,6-4 6-2
1,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Clement A.,Enqvist T.,Enqvist T.,56,5,-1,-1,-1.0,-1.0,3-6 3-6


In [2]:
import sqlite3

DB_PATH = Path("../tennis.db")
print("DB will be created at:", DB_PATH.resolve())

conn = sqlite3.connect(DB_PATH)

df.to_sql("matches_raw", conn, if_exists="replace", index=False)
long_df.to_sql("player_matches", conn, if_exists="replace", index=False)
features_agg.to_sql("player_features", conn, if_exists="replace", index=False)

conn.close()

print("Wrote tables to SQLite: matches_raw, player_matches, player_features")


DB will be created at: /Users/ardaogulcanuzer/Desktop/tennis_similarity_project/tennis.db
Wrote tables to SQLite: matches_raw, player_matches, player_features


In [3]:
def run_sql(query: str) -> pd.DataFrame:
    import sqlite3
    conn = sqlite3.connect(DB_PATH)
    out = pd.read_sql_query(query, conn)
    conn.close()
    return out


In [4]:
q1 = """
SELECT player, matches, win_rate
FROM player_features
WHERE season = 2025 AND surface = 'Hard'
ORDER BY win_rate DESC, matches DESC
LIMIT 20;
"""
run_sql(q1)


,player,matches,win_rate
0,Sinner J.,40,0.975000
1,Draper J.,16,0.875000
2,Alcaraz C.,41,0.853659
3,Djokovic N.,30,0.800000
4,Auger-Aliassime F.,50,0.760000
5,Rune H.,31,0.709677
6,Paul T.,23,0.695652
7,De Minaur A.,51,0.686275
8,Mensik J.,28,0.678571
9,Ruud C.,28,0.678571


In [5]:
q2 = """
SELECT player, surface,
       SUM(matches) AS total_matches,
       ROUND(AVG(win_rate), 4) AS avg_win_rate
FROM player_features
WHERE player IN ('Alcaraz C.', 'Sinner J.')
  AND season BETWEEN 2020 AND 2025
GROUP BY player, surface
ORDER BY player, total_matches DESC;
"""
run_sql(q2)


,player,surface,total_matches,avg_win_rate
0,Alcaraz C.,Hard,165,0.7549
1,Alcaraz C.,Clay,80,0.9047
2,Sinner J.,Hard,243,0.8087
3,Sinner J.,Clay,33,0.7167


In [6]:
q3 = """
SELECT player,
       SUM(matches) AS total_matches,
       ROUND(AVG(win_rate), 4) AS avg_win_rate_2020_2025
FROM player_features
WHERE surface = 'Hard'
  AND season BETWEEN 2020 AND 2025
GROUP BY player
HAVING total_matches >= 60
ORDER BY avg_win_rate_2020_2025 DESC
LIMIT 30;
"""
run_sql(q3)


,player,total_matches,avg_win_rate_2020_2025
0,Djokovic N.,154,0.8689
1,Sinner J.,243,0.8087
2,Alcaraz C.,165,0.7549
3,Medvedev D.,259,0.7499
4,Zverev A.,199,0.7367
5,Draper J.,88,0.7282
6,Monfils G.,129,0.6931
7,Rublev A.,260,0.6643
8,Berrettini M.,66,0.6481
9,Fritz T.,231,0.6477


In [7]:
q4 = """
SELECT player, matches, underdog_matches,
       ROUND(underdog_win_rate, 4) AS underdog_win_rate
FROM player_features
WHERE season = 2025 AND surface = 'Hard'
  AND underdog_matches >= 5
ORDER BY underdog_win_rate DESC, underdog_matches DESC
LIMIT 30;
"""
run_sql(q4)


,player,matches,underdog_matches,underdog_win_rate
0,Rune H.,31,6,0.6667
1,Monfils G.,20,7,0.5714
2,Medjedovic H.,18,11,0.5455
3,Griekspoor T.,29,13,0.5385
4,Munar J.,33,19,0.5263
5,Altmaier D.,31,24,0.5000
6,Mensik J.,28,10,0.5000
7,Fonseca J.,18,6,0.5000
8,Tien L.,38,17,0.4706
9,Davidovich Fokina A.,42,15,0.4667


In [8]:
q5 = """
SELECT pm.date, pm.tournament, pm.round, pm.surface,
       pm.player, pm.opponent, pm.is_win,
       pf.win_rate, pf.matches
FROM player_matches pm
JOIN player_features pf
  ON pm.player = pf.player
 AND pm.season = pf.season
 AND pm.surface = pf.surface
WHERE pm.player = 'Alcaraz C.'
  AND pm.season = 2025
  AND pm.surface = 'Hard'
ORDER BY pm.date DESC
LIMIT 25;
"""
run_sql(q5)


,date,tournament,round,surface,player,opponent,is_win,win_rate,matches
0,2025-11-16,Masters Cup,The Final,Hard,Alcaraz C.,Sinner J.,0,0.853659,41
1,2025-11-15,Masters Cup,Semifinals,Hard,Alcaraz C.,Auger-Aliassime F.,1,0.853659,41
2,2025-11-13,Masters Cup,Round Robin,Hard,Alcaraz C.,Musetti L.,1,0.853659,41
3,2025-11-11,Masters Cup,Round Robin,Hard,Alcaraz C.,Fritz T.,1,0.853659,41
4,2025-11-09,Masters Cup,Round Robin,Hard,Alcaraz C.,De Minaur A.,1,0.853659,41
5,2025-10-28,BNP Paribas Masters,2nd Round,Hard,Alcaraz C.,Norrie C.,0,0.853659,41
6,2025-09-30,Japan Open Tennis Championships,The Final,Hard,Alcaraz C.,Fritz T.,1,0.853659,41
7,2025-09-29,Japan Open Tennis Championships,Semifinals,Hard,Alcaraz C.,Ruud C.,1,0.853659,41
8,2025-09-28,Japan Open Tennis Championships,Quarterfinals,Hard,Alcaraz C.,Nakashima B.,1,0.853659,41
9,2025-09-27,Japan Open Tennis Championships,2nd Round,Hard,Alcaraz C.,Bergs Z.,1,0.853659,41


In [9]:
q5_result = run_sql(q5)
q5_result.head()


,date,tournament,round,surface,player,opponent,is_win,win_rate,matches
0,2025-11-16,Masters Cup,The Final,Hard,Alcaraz C.,Sinner J.,0,0.853659,41
1,2025-11-15,Masters Cup,Semifinals,Hard,Alcaraz C.,Auger-Aliassime F.,1,0.853659,41
2,2025-11-13,Masters Cup,Round Robin,Hard,Alcaraz C.,Musetti L.,1,0.853659,41
3,2025-11-11,Masters Cup,Round Robin,Hard,Alcaraz C.,Fritz T.,1,0.853659,41
4,2025-11-09,Masters Cup,Round Robin,Hard,Alcaraz C.,De Minaur A.,1,0.853659,41


In [10]:
OUT = Path("../outputs/tables")
OUT.mkdir(parents=True, exist_ok=True)

q5_result.to_csv(OUT / "q5_alcaraz_2025_hard_last25.csv", index=False)
print("Saved:", (OUT / "q5_alcaraz_2025_hard_last25.csv").resolve())

Saved: /Users/ardaogulcanuzer/Desktop/tennis_similarity_project/outputs/tables/q5_alcaraz_2025_hard_last25.csv
